# Http poller Feed creation

## Connect to a Velocity instance
For this example, we will connect to `a4iot-dev`

In [ ]:
from arcgis import GIS
from arcgis.realtime.velocity.feeds_manager import Feed

gis = GIS(
    url="https://devext.arcgis.com",
    username="pythontest_a4iot",
    password="v3locity.pyth0n",
)

velocity = gis.velocity

feeds = gis.velocity.feeds
feeds

## Configure the HTTP Poller Feed
Configuring a delimited format feed that contains location, time, track id and info about satellite locations.

Feed Location: http://websats.westus2.cloudapp.azure.com/websats/satellites

In [ ]:
from arcgis.realtime.velocity.feeds import HttpPoller
from arcgis.realtime.velocity.http_authentication_type import (
    NoAuth,
    BasicAuth,
    CertificateAuth,
)
from arcgis.realtime.velocity.input.format import DelimitedFormat
from arcgis.realtime.velocity.feeds.geometry import XYZGeometry, SingleFieldGeometry
from arcgis.realtime.velocity.feeds.time import TimeInterval, TimeInstant
from arcgis.realtime.velocity.feeds.run_interval import RunInterval


# HTTP Poller Properties
name = "http_poller_feed_1"
description = "some description about the http poller feed"
url = "http://websats.westus2.cloudapp.azure.com/websats/satellites"
http_auth = NoAuth()
# http_auth = BasicAuth(username="user1", password="123")
# http_auth = CertificateAuth(pfx_file_http_location="https://some.where", password="123")

http_headers = {}
# http_headers = {
#     "Content-Type": "application/json"
# }

url_params = {}
# url_params = {
#     "f": "json"
# }

http_poller = HttpPoller(
    label=name,
    description=description,
    url=url,
    http_method="GET",
    http_auth_type=http_auth,
    url_params=url_params,
    http_headers=http_headers,
    enable_long_polling=False,
    data_format=None
)



# # all properties can also be defined right away in the constructor as follows
data_format = DelimitedFormat("\n", ",", False)
# geometry = XYZGeometry(
#     x_field="category_longitude",
#     y_field="category_latitude",
#     wkid=4326,
#     z_field="category_altitude",
#     z_unit="Meters"
# )
#
# time = TimeInterval(
#     interval_start_field="pubDate",
#     interval_end_field="updated"
# )
#
# run_interval = RunInterval(
#     cron_expression="0 * * ? * * *",
#     timezone="America/Los_Angeles"
# )
#
# http_poller1 = HttpPoller(
#     label=name,
#     description=description,
#     url=url,
#     http_method="GET",
#     http_auth_type=http_auth,
#     track_id_field="link",
#     data_format=data_format,
#     geometry=geometry,
#     time=time,
#     run_interval=run_interval
# )

### Manipulate the schema - rename or remove fields, change field data-type
1. Renaming `title` to `updated_field`
2. Dropping `description`

In [ ]:
http_poller.rename_field("field1", "name")
http_poller.remove_field("field7")

### Set track id field
Set `link` as the track-id (Sorry, this feed doesn't have a good candidate for track-id!)

In [ ]:
http_poller.set_track_id("field2")

### Set time field
Time interval
1. Start time - `pubDate`
2. End time - `updated`

In [ ]:
# time interval
time = TimeInstant(time_field="field3")

# time instant
# time = TimeInstant(time_field="pubDate")


http_poller.set_time_config(time=time)

### Set geometry field
Configuring X,Y and Z fields

In [ ]:
geometry = XYZGeometry(
    x_field="field5",
    y_field="field6",
    wkid=4326
)

# a single field geometry could also be configured
# geometry = SingleFieldGeometry(
#     geometry_field="field6",
#     geometry_type="esriGeometryPoint",
#     geometry_format="esrijson",
#     wkid=4326
# )

http_poller.set_geometry_config(geometry=geometry)

### Set recurrence

In [ ]:
http_poller.run_interval = RunInterval(
    cron_expression="0 * * ? * * *", timezone="America/Los_Angeles"
)

### Create the Feed!

In [ ]:
feeds.create(http_poller)

feeds.items